# 05 — Tool Usage: Cosmos as a Strands Tool

**What this notebook demonstrates:**
- Using `cosmos_vision_invoke` as a standalone tool
- Using `cosmos_sysinfo` for diagnostics
- Using `video_probe` and `video_extract_frames` for media utilities
- Using `image_read` to load images as base64
- Composing tools together in a pipeline

**Key concept:** All 21 strands-cosmos tools can be used:
1. **Directly** — call the function with arguments
2. **Inside an Agent** — pass as `tools=[...]` to any Strands Agent

This notebook tests tools directly (no outer agent needed).

---

In [ ]:
import os
os.environ['QWEN_VL_VIDEO_READER'] = 'torchcodec'

import sys, os, time
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

project_root = os.path.abspath('..')
sample_video = os.path.join(project_root, 'sample.mp4')
sample_image = os.path.join(project_root, 'sample.png')
print(f"📹 Video: {sample_video} ({os.path.exists(sample_video)})")
print(f"🖼  Image: {sample_image} ({os.path.exists(sample_image)})")

## Tool 1: `cosmos_sysinfo` — System Diagnostics

Check GPU, platform, and thermal status.

In [ ]:
from strands_cosmos import cosmos_sysinfo

result = cosmos_sysinfo()
print(result['content'][0]['text'])

## Tool 2: `video_probe` — Video Metadata

Get duration, resolution, codec, fps — wraps `ffprobe`.

In [ ]:
from strands_cosmos import video_probe

result = video_probe(video_path=sample_video)
print(f"Status: {result['status']}")
print(result['content'][0]['text'][:800])

## Tool 3: `video_extract_frames` — Extract Frames

Pull frames from video at a given FPS.

In [ ]:
from strands_cosmos import video_extract_frames

result = video_extract_frames(
    video_path=sample_video,
    output_dir='/tmp/cosmos_frames',
    fps=1.0,
    max_frames=5
)
print(f"Status: {result['status']}")
print(result['content'][0]['text'])

In [ ]:
# Show extracted frames
import glob
from PIL import Image
from IPython.display import display

frames = sorted(glob.glob('/tmp/cosmos_frames/frame_*.jpg'))[:3]
for f in frames:
    img = Image.open(f)
    print(f"  {os.path.basename(f)}: {img.size}")
    display(img.resize((320, int(320 * img.size[1] / img.size[0]))))

## Tool 4: `image_read` — Load Image as Base64

Useful for passing images to APIs or embedding in messages.

In [ ]:
from strands_cosmos import image_read

result = image_read(image_path=sample_image)
print(f"Status: {result['status']}")
# Show first 100 chars of base64
text = result['content'][0]['text']
print(f"Base64 length: {len(text)} chars")
print(f"Preview: {text[:80]}...")

## Tool 5: `cosmos_vision_invoke` — Direct VLM Inference

The most powerful tool — invokes the full Cosmos-Reason2 VLM on video/image.

In [ ]:
from strands_cosmos import cosmos_vision_invoke

t0 = time.time()
result = cosmos_vision_invoke(
    prompt="Describe the scene briefly. What objects are visible?",
    video_path=sample_video,
    max_tokens=512,
)
print(f"Status: {result['status']}")
if result['status'] == 'success':
    print(f"\nResponse:\n{result['content'][0]['text']}")
print(f"\n⏱  Time: {time.time() - t0:.1f}s")

## Tool 6: `cosmos_reason_hf` — HF Direct Reasoning

Alternative to `cosmos_vision_invoke` — uses direct HF Transformers pipeline.

In [ ]:
from strands_cosmos import cosmos_reason_hf

t0 = time.time()
result = cosmos_reason_hf(
    prompt="What physics principles are shown in this video?",
    video_path=sample_video,
    max_new_tokens=256,
)
print(f"Status: {result['status']}")
print(result['content'][0]['text'][:500])
print(f"\n⏱  Time: {time.time() - t0:.1f}s")

---
## Summary: All Tools Tested

| Tool | Status | Notes |
|------|--------|-------|
| `cosmos_sysinfo` | ✅ | Platform diagnostics |
| `video_probe` | ✅ | ffprobe metadata |
| `video_extract_frames` | ✅ | Frame extraction |
| `image_read` | ✅ | Base64 encoding |
| `cosmos_vision_invoke` | ✅ | Full VLM inference |
| `cosmos_reason_hf` | ✅ | HF direct inference |

**TRT-only tools** (require Jetson with TRT-Edge-LLM built):
- `cosmos_inference`, `cosmos_serve`, `cosmos_quantize`, `cosmos_export_onnx`, `cosmos_build_engine`
- Test these with `just doctor` to see what's available on your host.

**Pipeline tools** (require companion repos cloned via `just setup`):
- `cosmos_predict_generate`, `cosmos_transfer_generate`, `cosmos_curate`, `cosmos_evaluate`
- `cosmos_post_train`, `cosmos_distill`